# Figs. 4–5 — Spatial SNN


In [ ]:
# Setup
# Load model parameters, patient event timings, and contact geometry.
# Patient FIT/CAL observations come from longer recordings than the raw-data demo.
import _setup
from modeling import computational as snn, panel_analysis as panels
from modeling.experiments import search_core_counts
from analysis import from_events as analysis, spectral_fields as spectral, model_transition as transition
from plotting import computational as plot
from workflows.tutorial import model_inputs, TutorialOutput, save_model_analysis

# Display figures inline after importing the model modules.
get_ipython().run_line_magic('matplotlib', 'inline')

config, patient = model_inputs()
out = TutorialOutput('snn')


In [ ]:
# Model definition
# Build a delayed spatial LIF network on a 20 x 20 mm sheet (80% E, 20% I).
# Cores lower local E thresholds and receive independent OU/Poisson input.
# The default configuration uses 40,000 neurons; settings are in config/snn_tutorial.json.
substrate = snn.build_substrate(config)


In [ ]:
# Fitting objective
# Compare rank patterns, local/inter-shaft timing, and contact participation.
# FIT defines the target distributions; independent CAL data scale their discrepancies.
# J averages the three calibrated terms without using TA/TB labels.
objective = snn.fit_objective(patient.fit, patient.cal, patient.names,
    n_fourier=config['n_fourier'], cal_draw_indices=patient.cal_draw_indices)


In [ ]:
# Fig. 4A: Baseline simulation
# Fix the core positions, integrate membrane and synaptic dynamics, and score the events.
# Display the threshold map, virtual contacts, and E/I population activity.
baseline = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=config['training_seed'])
snn.score_simulation(baseline, patient.observer, objective, config['burnin_ms'])
out.show(plot.substrate_panel(baseline, config['radius_mm']), 'fig4_A_model')
out.simulation('baseline_new_simulation', baseline)


In [ ]:
# Fig. 4F: Contact waveforms
# Spatially weight the new spikes and filter the readout at 30–80 Hz.
# This activity proxy has different units from SEEG voltage.
out.show(plot.bandpassed_model_waveform(baseline), 'fig4_F_baseline_waveforms')


In [ ]:
# Core optimization
# Search core positions and outward E-to-E weights with equal budgets for 1–4 cores.
# Each candidate requires a new simulation and J evaluation.
# The default fixes the connection angle; the full configuration can optimize it jointly.
searches = search_core_counts(substrate, patient.contacts, patient.observer, objective,
    out.path/'core_search', core_counts=(1, 2, 3, 4),
    budget=config['optimization_budget'], restarts=config['restarts'])
rows = [row for search in searches for row in search['rows']]


In [ ]:
# Fig. 4B: Observable errors
# Inspect mean-rank, within-shaft order, and participation errors for the candidates.
# These descriptive errors are separate from the distributional objective J.
out.show(plot.raw_errors(rows), 'fig4_B_observable_errors')


In [ ]:
# Fig. 4C: Core-count comparison
# Plot the best training J against evaluation count for each number of cores.
# The default two evaluations per core count illustrate the method with a small budget.
out.show(plot.optimization_curves(searches), 'fig4_C_core_search')


In [ ]:
# Fig. 4D: Direction sensitivity
# Rotate the E-to-E connection kernel, rebuild the network, and simulate again.
# Keep other parameters and noise fixed; reuse the baseline at zero angle offset.
angle_rows = snn.angle_response(config, config['centers_mm'], patient.contacts, patient.observer,
    objective, angles=[a for a in config['angle_values'] if a != 0])
angle_rows = sorted([dict(angle=0, **baseline['score'])] + angle_rows, key=lambda row: row['angle'])
out.show(plot.raw_errors(angle_rows, angle=True), 'fig4_D_direction_response')


In [ ]:
# Fig. 4E: Low-error core locations
# Pool core positions from the lowest 20% of finite-J candidate configurations.
out.show(plot.core_density(rows, config['radius_mm']), 'fig4_E_core_positions')


In [ ]:
# Fig. 4G: Patient and model rank profiles
# Fit patient templates, assign baseline model events, and compare their mean ranks.
# Nonparticipating contacts remain missing; this is a descriptive model comparison.
matching = panels.model_patient_comparison(baseline['primary_centroids_ms'], patient.fit,
    patient.names, patient.contact_contract, n_null=1000)
out.show(plot.model_rank_profiles(matching, patient.names), 'fig4_G_mean_rank')


In [ ]:
# Fig. 4H: Held-out contact matching
# Assign templates using one half of the contacts and score the other half.
# Swap the halves and combine the correlations.
out.show(plot.model_crossfit_matrix(matching), 'fig4_H_crossfit')


In [ ]:
# Supplementary Video 2: Propagation at two working points
# Keep the cores fixed and compare outward E-to-E weights of 1.0 and 1.25.
# Select the first eligible event of each patient-template class at each working point.
# Align each event to its own earliest centroid and display its waveform and spatial activity.
comparison_run = snn.simulate(substrate, config['centers_mm'], patient.contacts,
    outward=1.0, seed=config['training_seed'])
snn.score_simulation(comparison_run, patient.observer, objective, config['burnin_ms'])
patient_templates = analysis.cluster_events(panels.centroid_events(patient.fit, patient.names))
comparison, selection = panels.select_propagation_events([baseline, comparison_run],
    patient_templates['templates'], patient.names, config['burnin_ms'])
plot.selected_propagation_movie(comparison, selection, out.figures/'supplementary_video2_new.gif', config['radius_mm'])
out.simulation('outward_100', comparison_run)


In [ ]:
# Fig. 4I method: Independent noise
# Keep the baseline configuration and score a new noise realization.
# This is a single-patient sensitivity check; a cohort requires separate patient fits.
confirmation = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=928401)
snn.score_simulation(confirmation, patient.observer, objective, config['burnin_ms'])
out.show(plot.noise_confirmation(baseline, confirmation), 'fig4_I_noise_confirmation')
out.simulation('new_noise_confirmation', confirmation)


In [ ]:
# Fig. 5A: Slow-variable simulation
# Keep the network and input settings; enable inhibitory-efficacy Z and adaptation M.
# Simulate again and record the synaptic-current proxy.
# Compute the display onset and the sustained high-rate entry separately.
zm = snn.simulate(substrate, config['centers_mm'], patient.contacts, seed=config['training_seed'],
    zm=config['zm'], record_current=True, duration_ms=18000.)
onset = transition.display_entry(zm)
entry = spectral.native_entry(zm['time_ms'], zm['rate_E'], dt_ms=config['parameters']['dt'])
out.show(plot.raw_current_waveforms(zm, onset=onset), 'fig5_A_waveforms')
out.simulation('zm_enabled', zm)


In [ ]:
# Fig. 5B: Slow variables
# Plot the mean Z and effective adaptation current eta*M across E cells.
out.show(plot.slow_variables(zm, config['zm']['eta_m']), 'fig5_B_slow_variables')


In [ ]:
# Fig. 5C: Spatial activity
# Average E activity over 50 ms at four fixed times around entry.
# Use evenly spaced times when no entry is detected.
out.show(plot.activity_snapshots(zm, config['radius_mm'], onset=onset), 'fig5_C_spatial_activity')


In [ ]:
# Fig. 5D: State trajectory
# Pair Z, eta*M, and E firing rate along the same simulation time axis.
# A single trajectory describes the transition but does not establish an attractor.
out.show(plot.state_trajectory(zm, config['zm']['eta_m']), 'fig5_D_state_trajectory')


In [ ]:
# Fig. 5E: Onset energy field
# Fix the projection geometry using the patient interictal template.
# Compute 1–150 Hz power in the first second after entry, normalize to baseline
# as robust-z, and project onto the fixed plane.
energy = spectral.energy_field_trajectory(zm['contact_current'], 2000., patient.contacts,
    patient_templates['templates'][0], patient_templates['support'][0], baseline_sec=(1.5, 3.), band=(30, 80))
onset_field = transition.onset_broadband_field(zm['contact_current'], 2000., onset)
out.show(plot.onset_energy_field(zm, onset_field, energy), 'fig5_E_model_onset_field')


In [ ]:
# Supplement: Time-varying field scores
# Track 30–80 Hz energy and its correlation/projection onto the fixed interictal field.
out.show(plot.energy_alignment(energy), 'supplementary_gamma_field_scores')


In [ ]:
# Save model results
# Save objective settings, matching scores, event selection, and energy-field results.
# Full clinical comparisons and the historical optimization protocol are described
# in METHODS_AND_PANELS.md.
save_model_analysis(out, config, objective, matching, selection, onset, entry, onset_field, energy,
    angle_rows=angle_rows)
